## Day 19: Time-Based Retention Analysis

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("../data/meetmux_transactions.csv")
df["PurchaseDate"] = pd.to_datetime(df["PurchaseDate"])

df["OrderMonth"] = df["PurchaseDate"].dt.to_period("M")
df["CohortMonth"] = (
    df.groupby("CustomerID")["PurchaseDate"].transform("min").dt.to_period("M")
)

def get_date_int(df, column):
    year = df[column].dt.year
    month = df[column].dt.month
    return year, month

order_year, order_month = get_date_int(df, "OrderMonth")
cohort_year, cohort_month = get_date_int(df, "CohortMonth")

years_diff = order_year - cohort_year
months_diff = order_month - cohort_month

df["CohortIndex"] = years_diff * 12 + months_diff + 1

df[["CustomerID", "CohortMonth", "CohortIndex"]].head()

In [ ]:
cohort_data = df.groupby(["CohortMonth", "CohortIndex"])["CustomerID"].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index="CohortMonth", columns="CohortIndex", values="CustomerID")

cohort_sizes = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_sizes, axis=0)

plt.figure(figsize=(12, 8))
sns.heatmap(retention, annot=True, fmt=".0%", cmap="YlGnBu")
plt.title("MeetMux User Retention Cohorts")
plt.show()

retention

In [ ]:
retention_by_index = retention.mean(axis=0, skipna=True)
cliff_index = (retention_by_index.shift(1) - retention_by_index).idxmax()
cliff_drop = (retention_by_index.shift(1) - retention_by_index).max()

month3_avg = retention[3].mean(skipna=True) if 3 in retention.columns else np.nan

long_term_col = int(retention.columns.max())
highest_long_term_cohort = retention[long_term_col].idxmax()
highest_long_term_value = retention.loc[highest_long_term_cohort, long_term_col]

cliff_index, cliff_drop, month3_avg, long_term_col, highest_long_term_cohort, highest_long_term_value

If the January Cohort has 40% retention in Month 3, but the March Cohort only has 15% in Month 3, it suggests the March acquisitions were lower quality (weaker intent/fit) or that something changed around March (pricing, onboarding, product stability, campaign targeting) that reduced early value realization and increased churn.